In [1]:
import sys
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.CRITICAL, force=True, stream=sys.stdout)
# logging.getLogger('torch_numopt.solve_system').setLevel(logging.DEBUG)
# logging.
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.GradientDescent(model.parameters(), lr_method="lipschitz")
# opt = torch_numopt.GradientDescentLipschitz(model.parameters())
# opt = torch_numopt.ConjugateGradient(model.parameters())
# opt = torch_numopt.LBFGS(model.parameters())
# opt = torch_numopt.AdaHessian(model.parameters())"
# opt = torch_numopt.GaussNewton(model.parameters(), damping="identity", mu=1e-4)
# opt = torch_numopt.LevenbergMarquardt(model.parameters())
# opt = torch_numopt.NewtonCG(model.parameters(), lr_init=1e-2)
# opt = torch_numopt.Newton(model.parameters(), lr_init=1e-2)
# opt = torch_numopt.Newton(model.parameters(), lr_init=1e-3, lr_method="lipschitz")

# opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_method="bisect")
# opt = torch_numopt.ConjugateGradientLS(model.parameters(), line_search_cond="armijo", line_search_method="backtrack")
# opt = torch_numopt.LBFGSLS(model.parameters())
# opt = torch_numopt.AdaHessianLS(model.parameters())
# opt = torch_numopt.GaussNewtonLS(model.parameters())
# opt = torch_numopt.LevenbergMarquardtLS(model.parameters())
# opt = torch_numopt.NewtonCGLS(model.parameters())
# opt = torch_numopt.NewtonLS(model.parameters())

# opt = torch_numopt.GradientDescentTR(model.parameters(), trust_region_method="cauchy")
# opt = torch_numopt.AdaHessianTR(model.parameters())
# opt = torch_numopt.GaussNewtonTR(model.parameters(), trust_region_method="dogleg")
# opt = torch_numopt.LevenbergMarquardtLS(model.parameters())
opt = torch_numopt.NewtonTR(model.parameters(), trust_region_method="cauchy")

# opt = torch_numopt.NumericalOptimizer(
#     model.parameters(),
#     curvature_estimator=torch_numopt.ExactHessianCalculator(),
#     lr_init=0.5,
#     solver="cg-trunc"
# )
# opt = torch_numopt.LineSearchOptimizer(
#     model.parameters(),
#     lr_init=1,
#     curvature_estimator=torch_numopt.ExactBlockHessianCalculator(),
#     line_search=torch_numopt.create_line_search_solver(
#         condition="armijo",
#         method="backtrack",
#         c1 = 1e-4,
#         c2 = 0.9,
#         tau = 0.1,
#         max_iter = 20,
#         tol = 1e-8,
#     ),
#     solver="cg",
# )
curv_est = torch_numopt.ExactHessianCalculator()
# curv_est = torch_numopt.GaussNewtonApproximation()
# curv_est = torch_numopt.GaussNewtonBlockApproximation()
opt = torch_numopt.TrustRegionOptimizer(
    model.parameters(),
    radius_init=1.0,
    trust_region=torch_numopt.trust_region.ExactTRSolver(curv_est)
    # trust_region=torch_numopt.trust_region.DoglegTRSolver(curv_est)
)

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.25722536444664
epoch:  1, loss: 0.165870800614357
epoch:  2, loss: 0.10999080538749695
epoch:  3, loss: 0.07081414759159088
epoch:  4, loss: 0.03323724493384361
epoch:  5, loss: 0.03323724493384361
epoch:  6, loss: 0.03020217828452587
epoch:  7, loss: 0.029736200347542763
epoch:  8, loss: 0.029299601912498474
epoch:  9, loss: 0.02723909355700016
epoch:  10, loss: 0.023686803877353668
epoch:  11, loss: 0.01724141649901867
epoch:  12, loss: 0.01724141649901867
epoch:  13, loss: 0.015274347737431526
epoch:  14, loss: 0.013516649603843689
epoch:  15, loss: 0.01198706403374672
epoch:  16, loss: 0.009654240682721138
epoch:  17, loss: 0.008232802152633667
epoch:  18, loss: 0.007798910140991211
epoch:  19, loss: 0.007663215044885874
epoch:  20, loss: 0.0076306769624352455
epoch:  21, loss: 0.007463949732482433
epoch:  22, loss: 0.007383470423519611
epoch:  23, loss: 0.007264693733304739
epoch:  24, loss: 0.007145426701754332
epoch:  25, loss: 0.006990264635533094
epoch:  26,

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9988306760787964
Test metrics:  R2 = 0.9978476166725159
